In [1]:
%pip install faster-whisper

Note: you may need to restart the kernel to use updated packages.


In [9]:
from faster_whisper import WhisperModel

model = WhisperModel("small", device="cpu", compute_type="int8")

In [10]:
print(dir(model))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_get_feature_kwargs', '_split_segments_by_timestamps', 'add_word_timestamps', 'detect_language', 'encode', 'feat_kwargs', 'feature_extractor', 'find_alignment', 'frames_per_second', 'generate_segments', 'generate_with_fallback', 'get_prompt', 'hf_tokenizer', 'input_stride', 'logger', 'max_length', 'model', 'num_samples_per_token', 'supported_languages', 'time_precision', 'tokens_per_second', 'transcribe']



9 seconds to transcribe 25 second audio

In [11]:
segments, _ = model.transcribe("audio_test_1.ogg")

text = " ".join([seg.text for seg in segments])
print(text)

 Apu raho, mutu kwa daizi, dumu niya rgwanda kazi, muluka murgwanda  Iji wugu cha jitemba matamu wuichi, iji wugu timi sozi di mumbi  Iji wugu cha weza mijizabdinshi, dukunda wa shitsi  Dawa raritsen, murakaza meza mubi sanga, murgwanda lgwadsh  Mwakoza kyan


In [ ]:
%pip install transformers soundfile torch

In [14]:
from transformers import VitsModel, AutoTokenizer
import torch
import soundfile as sf
from IPython.display import Audio

# Load Kinyarwanda TTS model
model = VitsModel.from_pretrained("facebook/mms-tts-kin")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-kin")

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

In [15]:
# text = "Muraho, iyi ni ikizamini cya text to speech."
# text = "Muraho neza, amakuru yawe? Uyu munsi ubusitani bwawe bumeze neza."
text = " Apu raho, mutu kwa daizi, dumu niya rgwanda kazi, muluka murgwanda  Iji wugu cha jitemba matamu wuichi, iji wugu timi sozi di mumbi  Iji wugu cha weza mijizabdinshi, dukunda wa shitsi  Dawa raritsen, murakaza meza mubi sanga, murgwanda lgwadsh  Mwakoza kyan"

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform

# Convert to numpy
audio = output.squeeze().cpu().numpy()

# Save (optional)
sf.write("output.wav", audio, samplerate=16000)

# Play in notebook
Audio(audio, rate=15000)

In [ ]:
from fastapi import FastAPI, UploadFile
import tempfile

app = FastAPI()

@app.post("/voice")
async def voice(file: UploadFile):
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        tmp.write(await file.read())
        path = tmp.name

    # ASR
    segments, _ = model.transcribe(path)
    text = " ".join([seg.text for seg in segments])

    # Intent
    intent = detect_intent(text)

    # Fake sensor data (replace later)
    sensor_data = {"moisture": 25, "temperature": 30}

    # Response
    response = handle_intent(intent, sensor_data)

    return {
        "text": text,
        "intent": intent,
        "response": response
    }